# XGBoost Model Training for Trip Distance Prediction
This notebook loads the prepared features, fine-tunes an XGBoost regressor, and evaluates performance.

In [5]:
# Install required packages if missing
try:
    import xgboost
except ImportError:
    !pip install xgboost
    import xgboost

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [6]:
# Load features and target
df_X = pd.read_csv('X_features_distance.csv')
df_y = pd.read_csv('y_target_distance.csv')

# Merge to align rows
train_df = pd.merge(df_X, df_y, on=['VehicleID', 'trip_id'])

# Drop ID columns for training
X = train_df.drop(['VehicleID', 'trip_id', 'total_trip_distance_km'], axis=1)
y = train_df['total_trip_distance_km']

# Convert H3 zone columns to integers
X['h3_start'] = X['h3_start'].apply(lambda x: int(x, 16))
X['h3_end'] = X['h3_end'].apply(lambda x: int(x, 16))


In [7]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# XGBoost hyperparameter grid for fine-tuning
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb = xgboost.XGBRegressor(objective='reg:squarederror', random_state=42)
grid_search = GridSearchCV(xgb, param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

print('Best parameters:', grid_search.best_params_)
print('Best CV MAE:', -grid_search.best_score_)

Fitting 3 folds for each of 72 candidates, totalling 216 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=1.0; total time=   1.3s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=0.8; total time=   1.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=1.0; total time=   1.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=0.8; total time=   1.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=1.0; total time=   1.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=1.0; total time=   1.3s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=0.8; total time=   1.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=1.0; total time=   1.4s
[CV] END c

In [9]:
# Evaluate on test set
best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Test MAE: {mae:.2f}")
print(f"Test RMSE: {rmse:.2f}")
print(f"Test R2: {r2:.3f}")

Test MAE: 3.51
Test RMSE: 8.69
Test R2: 0.839


In [10]:
best_xgb.save_model('best_xgb_trip_distance.model')

/Users/jul/.pyenv/versions/3.11.6/lib/python3.11/site-packages/xgboost/sklearn.py:1028: UserWarning: [20:03:13] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)


## The best XGBoost model is trained and evaluated. You can now save the model or analyze feature importance.